# ARC2 012 kernel probe

Local preparation only; publication and execution are separate. Plan 012. Fixed Qwen3.8-27B BF16, Kaggle L4x4, no internet. K mounts the new wheel Dataset, P does not; no S4 adapter.


### 1. Restore hashed code and task packets (scoring excluded from K)


In [ ]:
import os
os.environ['HF_DEACTIVATE_ASYNC_LOAD']='1'
import base64, hashlib, io, json, pathlib, time
import zipfile
NOTEBOOK_START=time.monotonic()
CODE=pathlib.Path('/kaggle/working/012/kernel-probe'); CODE.mkdir(parents=True,exist_ok=True)
CODE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '10482e0987fd83cee877882d665ff0d71dec5ca6ec8c5433c4cf46d6cfb94cba', 'p2-challenges.json': 'abef4b67006970e7a899e9a15f661d65ec56b6f3f4264bf5c667717f916804d3', 'p2-queries-291dc1e1.json': 'ceef17766c86b944898d9bdd84b36ab69079947519db07aa7a6311c7412ca287', 'p2-queries-db118e2a.json': '836962766952589c5a07600f43d047f3a636c34fece0d4badec9a0b1c916ed96', 'p2-queries.json': '6379a22f3581b32411b7547f187c57403dbd6ba07fa22875d3b6e4a63dbd8b61', 'probe_gdn_inference.py': '33f6fbfd702f34542ae4927fe7c9d0e07c977d8600a9eb2978d114c33166d85f', 'run_budgeted_grid_experiment.py': '10178b771eadbc44009032dd440c53604724a930fcf17dc1c43ca00e4e7bc287', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': '03aaa3818a8f3beafee019140bae823f968e913af72ef297fa7531f24816a37a', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
CORE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '10482e0987fd83cee877882d665ff0d71dec5ca6ec8c5433c4cf46d6cfb94cba', 'probe_gdn_inference.py': '33f6fbfd702f34542ae4927fe7c9d0e07c977d8600a9eb2978d114c33166d85f', 'run_budgeted_grid_experiment.py': '10178b771eadbc44009032dd440c53604724a930fcf17dc1c43ca00e4e7bc287', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': '03aaa3818a8f3beafee019140bae823f968e913af72ef297fa7531f24816a37a', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
CORE_MANIFEST_SHA='31ce366d4df03d344cfd5ef1cbc8ed747865208eba97841acbac680de30ae14f'
payload=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert set(z.namelist())==set(CODE_MANIFEST)
    for name in z.namelist():
        data=z.read(name)
        assert hashlib.sha256(data).hexdigest()==CODE_MANIFEST[name]
        (CODE/name).write_bytes(data)
(CODE/'source-manifest.json').write_text(json.dumps(CODE_MANIFEST,indent=2))
(CODE/'core-manifest.json').write_text(json.dumps(CORE_MANIFEST,indent=2))
print('012_CORE_SHA',CORE_MANIFEST_SHA)


### 2. Run installation and one-load probe under the 30-minute parent deadline


In [ ]:
import subprocess
telemetry_file=open(CODE/'gpu-telemetry.csv','w')
telemetry=subprocess.Popen(['nvidia-smi','--query-gpu=timestamp,index,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw','--format=csv','-l','10'],stdout=telemetry_file,stderr=subprocess.STDOUT)
try:

    receipts=[]
    for p in pathlib.Path('/kaggle/input').rglob('receipt.json'):
        try:
            value=json.loads(p.read_text())
            if value.get('core_manifest_sha256')==CORE_MANIFEST_SHA and value.get('status')=='passed':
                receipts.append(str(p))
        except (ValueError,OSError):
            pass
    assert receipts,'Matching full CPU receipt must be mounted before GPU work'
    print('MATCHED_CPU_RECEIPT',receipts[0],flush=True)

    import sys
    sys.path.insert(0,str(CODE))
    from pathlib import Path
    import hashlib,json,sys
    def digest(p):
        h=hashlib.sha256()
        f=open(p,'rb')
        b=f.read(2097152)
        while b:
            h.update(b);b=f.read(2097152)
        f.close();return h.hexdigest()
    models=[p.parent for p in Path('/kaggle/input').rglob('config.json') if 'qwen3' in p.read_text().lower()]
    if len(models)!=1:
        raise RuntimeError('Expected one Qwen3.8 model dir, found %d'%len(models))
    model_dir=str(models[0])
    print('MODEL_DIR',model_dir)
    assert not list(Path('/kaggle/input').rglob('adapter_step_25/adapter_model.safetensors')),'012 mounts no S4 adapter'
    sys.path.insert(0,str(CODE))

    setup_path=CODE/'setup-k.py'
    setup_path.write_text("import time\nPREP_DEADLINE=NOTEBOOK_START+120\nimport glob,subprocess,sys,hashlib,pathlib,shutil,json\ncand=[p for p in glob.glob('/kaggle/input/**/*.whl',recursive=True) if 'qwen38' not in p]\nwant=[{'file': 'causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl', 'version': '1.7.0+cu12torch2.10cxx11abiTRUE', 'sha256': '8e81f8c76435ad31aa41edc6c0c9d26de971a293b190183d2816da71547614d7', 'target': 'cp312-linux_x86_64, CXX11 ABI TRUE, cu12/torch2.10', 'source': '011 gdn-followup-evidence.md 2026-10-11; pinned, no version scan'}, {'file': 'fla_core-0.5.2-py3-none-any.whl', 'version': '0.5.2', 'sha256': '5e830c85bad3d0d34677f98ac7074d08687a3756f0f0499d95ceb96eb6920761', 'target': 'py3-none-any', 'source': '011 gdn-followup-evidence.md 2026-10-11; pinned, no version scan'}, {'file': 'flash_linear_attention-0.5.2-py3-none-any.whl', 'version': '0.5.2', 'sha256': 'dcf405d81f5426393b59037097aa700d0f4a841465d5028d5aa543f4502f2400', 'target': 'py3-none-any', 'source': '011 gdn-followup-evidence.md 2026-10-11; pinned, no version scan'}]\ndef _norm(n):\n    return n.replace('+','')\nbyexact={pathlib.Path(p).name:p for p in cand}\nbynorm={_norm(pathlib.Path(p).name):p for p in cand}\nresolved={}\nfor w in want:\n    if w['file'] in byexact:\n        resolved[w['file']]=byexact[w['file']]\n    elif _norm(w['file']) in bynorm:\n        resolved[w['file']]=bynorm[_norm(w['file'])]\n        print('WHEEL_NAME_SANITIZED',w['file'],'->',pathlib.Path(resolved[w['file']]).name)\n    else:\n        raise RuntimeError('Missing K wheels in new Dataset: '+w['file'])\nwdir=pathlib.Path('/kaggle/working/wheels');wdir.mkdir(parents=True,exist_ok=True)\nmapping={}\nfor w in want:\n    h=hashlib.sha256(open(resolved[w['file']],'rb').read()).hexdigest()\n    assert h==w['sha256'],w['file']\n    print('WHEEL_SHA_OK',pathlib.Path(resolved[w['file']]).name,h[:16])\n    canon=wdir/w['file']\n    shutil.copyfile(resolved[w['file']],canon)\n    assert hashlib.sha256(open(canon,'rb').read()).hexdigest()==w['sha256']\n    mapping[pathlib.Path(resolved[w['file']]).name]=canon.name\n    print('WHEEL_RESTORED',canon.name)\n(CODE/'wheel-name-map.json').write_text(json.dumps(mapping,indent=2))\nprint('WHEEL_NAME_MAP',json.dumps(mapping))\np=subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps']+[str(wdir/w['file']) for w in want],capture_output=True,text=True,timeout=max(1,PREP_DEADLINE-time.monotonic()))\nprint((p.stdout or '')[-1200:])\nif p.returncode:\n    print((p.stderr or '')[-1200:]);raise RuntimeError('K wheel install failed')\nimport glob, subprocess, sys, json\nwheel=glob.glob('/kaggle/input/**/qwen38-wheelhouse-py312',recursive=True)\nif len(wheel)!=1:\n    raise RuntimeError('Expected one frozen wheelhouse, found %d'%len(wheel))\np=subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links='+wheel[0],'transformers==5.8.0','accelerate==1.13.0','peft==0.19.1'],capture_output=True,text=True,timeout=max(1,min(240,PREP_DEADLINE-time.monotonic())))\nprint((p.stdout or '')[-1200:])\nif p.returncode:\n    print((p.stderr or '')[-1200:]);raise RuntimeError('Offline base install failed')\nimport transformers,torch,peft,accelerate\nversions=dict(python=sys.version.split()[0],transformers=transformers.__version__,torch=torch.__version__,peft=peft.__version__,accelerate=accelerate.__version__)\nprint('VERSIONS',json.dumps(versions))\nassert transformers.__version__=='5.8.0'\nfrom pathlib import Path\nimport hashlib,json,sys\ndef digest(p):\n    h=hashlib.sha256()\n    f=open(p,'rb')\n    b=f.read(2097152)\n    while b:\n        h.update(b);b=f.read(2097152)\n    f.close();return h.hexdigest()\nmodels=[p.parent for p in Path('/kaggle/input').rglob('config.json') if 'qwen3' in p.read_text().lower()]\nif len(models)!=1:\n    raise RuntimeError('Expected one Qwen3.8 model dir, found %d'%len(models))\nmodel_dir=str(models[0])\nprint('MODEL_DIR',model_dir)\nassert not list(Path('/kaggle/input').rglob('adapter_step_25/adapter_model.safetensors')),'012 mounts no S4 adapter'\nsys.path.insert(0,str(CODE))\nimport json\nfrom transformers import AutoTokenizer\nimport arc_episode_data as episodes\nimport probe_gdn_inference as probe\ntok=AutoTokenizer.from_pretrained(model_dir,local_files_only=True,trust_remote_code=False)\ntasks=json.loads((CODE/'p2-challenges.json').read_text())\ntask=tasks['db118e2a']\nmsgs=episodes.build_messages(task['train'],task['test'][0]['input'],formatter='compact')\nprompt_text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)\npinfo=probe.write_f1_prompt_with_tokenizer(tok,prompt_text,str(CODE/'db118e2a-query0-f1-prompt.txt'))\nprint('PROMPT_RENDERED',json.dumps(pinfo))\nconfig={'notebook':'kernel-probe','task':'db118e2a/query0','prompt':'f1_full_nonhthinking',\n 'fixed_decode_tokens':64,'order':['A1','B1','B2','A2'],\n 'thresholds':{'a_rel_rms':0.0001,'b_rel_floor':0.01,'gain':0.15},\n 'notebook_seconds':1800,'load_limit_s':600,'save_window_s':120,\n 'model_dir':model_dir,'output_dir':'/kaggle/working/012/kernel-probe/output',\n 'prompt_file':'db118e2a-query0-f1-prompt.txt','prompt_sha256':pinfo['sha256'],\n 'prompt_tokens':pinfo['n_tokens'],'journal':'/kaggle/working/012/kernel-probe/journal.jsonl'}\n(CODE/'run-k.json').write_text(json.dumps(config,indent=2))\nprint(json.dumps(config,indent=2))\nassert not (CODE/'p2-scoring.json').exists(),'K must not stage scoring'\nassert (CODE/'probe_gdn_inference.py').is_file(),'packed probe entry missing'\nimport subprocess,sys\nprint('PROBE_IMPORTED',probe.__file__)\np=subprocess.run([sys.executable,probe.__file__,'--dry-run-cpu','--prompt-file',pinfo['path']],capture_output=True,text=True,timeout=60)\nprint((p.stdout or '')[-1500:])\nassert p.returncode==0,'probe --dry-run-cpu failed: '+(p.stderr or '')[-500:]\nprint('PROBE_DRY_RUN_OK')\n")
    import probe_gdn_inference as probe
    rc=probe.main(['--model-dir',model_dir,'--output-dir',str(CODE/'output'),'--prompt-file',str(CODE/'db118e2a-query0-f1-prompt.txt'),'--journal',str(CODE/'journal.jsonl'),'--notebook-start-monotonic',str(NOTEBOOK_START),'--setup-script',str(setup_path)])
    assert rc==0,rc

finally:
    telemetry.terminate()
    try: telemetry.wait(timeout=5)
    except subprocess.TimeoutExpired:
        telemetry.kill(); telemetry.wait(timeout=5)
    telemetry_file.close()


### Interpretation

CPU/K/P evidence stays separate. P2 is development data, not unseen validation. No score is claimed here.
